# TopicGPT Tuning: c-TF-IDF & Topic Filtering Optimization

This notebook tunes the c-TF-IDF and topic filtering parameters for TopicGPT's
assignment stage. It **reuses** existing assignment checkpoints from the modeling
phase and grid-searches over:

- **`max_df`** — c-TF-IDF maximum document frequency threshold
- **`min_docs`** — Minimum documents per topic to keep

Each subject uses its best sentence transformer model identified during modeling.

In [1]:
import os
import gc
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import product
from tqdm import tqdm
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.feature_extraction.text import TfidfVectorizer
from itertools import combinations
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Configuration

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
VERSION = "v1"

BASE_DIR = Path("../../../../data/preprocess")
CHECKPOINT_DIR = Path("../../../../models/topicGpt")
RESULT_DIR = Path("../../../../results/topicGpt/tunning")
TUNING_CKPT_DIR = CHECKPOINT_DIR  # tuning checkpoints go alongside modeling ones
EMBEDDING_DIR = Path("../../../../embedding")

# Best embedding model per subject (from modeling phase)
BEST_MODEL_MAP = {
    "cs":      "sentence_transformers_all_MiniLM_L6_v2",
    "math":    "sentence_transformers_all_MiniLM_L6_v2",
    "physics":  "all_distilroberta_v1",
}

# --- Tuning grid ---
MAX_DF_VALUES   = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
MIN_DOCS_VALUES = [10, 25, 50, 100, 150, 200, 250]

TOP_N_WORDS = 10
RBO_P = 0.9

# Create output dirs
for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Subjects: {LIST_SUBJECT}")
print(f"max_df grid:   {MAX_DF_VALUES}")
print(f"min_docs grid: {MIN_DOCS_VALUES}")
print(f"Total combos per subject: {len(MAX_DF_VALUES) * len(MIN_DOCS_VALUES)}")
print(f"Results dir: {RESULT_DIR.resolve()}")

Subjects: ['cs', 'math', 'physics']
max_df grid:   [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
min_docs grid: [10, 25, 50, 100, 150, 200, 250]
Total combos per subject: 42
Results dir: /home/nedo/Kuliah/TA/Program/results/topicGpt/tunning


## Checkpoint Utilities

In [3]:
def save_checkpoint(data, name: str, subject: str):
    """Save checkpoint to disk."""
    path = TUNING_CKPT_DIR / subject / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"  Checkpoint saved: {path}")

def load_checkpoint(name: str, subject: str):
    """Load checkpoint from disk, return None if not found."""
    path = TUNING_CKPT_DIR / subject / f"{name}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  Checkpoint loaded: {path}")
        return data
    return None

## Data Loading

In [4]:
def load_dataset(subject: str) -> pd.DataFrame:
    """Load preprocessed dataset."""
    file_path = BASE_DIR / subject / "emb" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)
    return df

all_data = {}
for subject in LIST_SUBJECT:
    df = load_dataset(subject)
    all_data[subject] = df
    print(f"{subject}: {len(df):,} documents loaded")

print(f"\nAll subjects loaded.")

cs: 165,756 documents loaded
math: 157,085 documents loaded
physics: 146,311 documents loaded

All subjects loaded.


## Generate Base Assignments

Instead of loading filtered assignment checkpoints from the modeling phase, we dynamically recompute the base assignments using the best sentence transformer for each subject.
This ensures zero filtering (`min_docs=0`) before the tuning loop begins.

In [5]:
MODEL_HF_MAP = {
    "all_distilroberta_v1": "all-distilroberta-v1",
    "sentence_transformers_all_MiniLM_L6_v2": "sentence-transformers/all-MiniLM-L6-v2"
}

# def load_doc_embeddings(subject: str, model_name: str):
#     path = EMBEDDING_DIR / subject / "emb" / f"{model_name}.npy"
#     if path.exists():
#         return np.load(path)
#     return None
def load_doc_embeddings(subject: str, model_name: str) -> np.ndarray:
    meta_path = EMBEDDING_DIR / subject / f"{model_name}_meta_v1.npy"
    if not meta_path.exists():
        meta_path = EMBEDDING_DIR / subject / f"{model_name}_v1_meta.npy"
    meta  = np.load(meta_path, allow_pickle=True).item()
    n, d  = meta["n_samples"], meta["emb_dim"]
    return np.memmap(
        EMBEDDING_DIR / subject / f"{model_name}_v1.mmap",
        dtype="float32", mode="r", shape=(n, d)
    )

def assign_topics_st(doc_embs, enriched_topics: dict, model_name: str, batch_size=2000, threshold=0.5):
    """Cosine-similarity assignment using a SentenceTransformer model with outlier thresholding."""
    hf_name  = MODEL_HF_MAP.get(model_name, model_name)
    st_model = SentenceTransformer(hf_name)
    
    topic_ids  = sorted(enriched_topics.keys())
    descs      = [enriched_topics[t].get("enriched_description") or enriched_topics[t]["description"]
                  for t in topic_ids]
    
    topic_embs = st_model.encode(descs, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    
    rows = []
    for start in tqdm(range(0, len(doc_embs), batch_size), desc=f"Assigning ({model_name})", leave=False):
        batch = np.array(doc_embs[start:start+batch_size])
        
        sims  = cos_sim(batch, topic_embs)
        best  = sims.argmax(axis=1)
        scores = sims.max(axis=1)
        
        for idx, (bi, sc) in enumerate(zip(best, scores)):
            if sc >= threshold:
                tid = topic_ids[bi]
                label = enriched_topics[tid]["label"]
            else:
                tid = -1
                label = "Outlier / Unassigned"
                
            rows.append({
                "doc_idx": start + idx, 
                "topic_id": tid,
                "topic_label": label,
                "confidence": float(sc)
            })
    return pd.DataFrame(rows)

all_base_assignments = {}

for subject in LIST_SUBJECT:
    best_model = BEST_MODEL_MAP[subject]
    
    # 1. Load document embeddings
    doc_embs = load_doc_embeddings(subject, best_model)
    if doc_embs is None:
        print(f"  ERROR: No doc embeddings found for {subject}/{best_model}")
        continue
        
    # 2. Load enriched topics
    enriched_ckpt = load_checkpoint("enrichment", subject)
    if not enriched_ckpt:
        print(f"  ERROR: No enriched topics found for {subject}")
        continue
        
    enriched_topics = enriched_ckpt["enriched_topics"]
    
    # 3. Assign topics
    print(f"Reassigning {subject} using {best_model}...")
    df_assign = assign_topics_st(doc_embs, enriched_topics, best_model)
    
    all_base_assignments[subject] = df_assign
    n_docs = len(df_assign)
    n_topics = df_assign["topic_id"].nunique()
    print(f"  {subject}: {n_docs:,} assignments, {n_topics} unique topics")

print(f"\nGenerated base assignments for {len(all_base_assignments)}/{len(LIST_SUBJECT)} subjects.")

  Checkpoint loaded: ../../../../models/topicGpt/cs/enrichment.pkl
Reassigning cs using sentence_transformers_all_MiniLM_L6_v2...


  cs: 165,756 assignments, 617 unique topics
  Checkpoint loaded: ../../../../models/topicGpt/math/enrichment.pkl
Reassigning math using sentence_transformers_all_MiniLM_L6_v2...


  math: 157,085 assignments, 611 unique topics
  Checkpoint loaded: ../../../../models/topicGpt/physics/enrichment.pkl
Reassigning physics using all_distilroberta_v1...


  physics: 146,311 assignments, 498 unique topics

Generated base assignments for 3/3 subjects.


## Tuning Functions

In [6]:
def filter_and_reindex_topics(df_assign: pd.DataFrame, min_docs: int) -> pd.DataFrame:
    """Filter topics with fewer than min_docs documents and reindex."""
    df_filtered = df_assign[df_assign["topic_id"] != -1].copy()

    topic_counts = df_filtered["topic_id"].value_counts()
    valid_topics = topic_counts[topic_counts >= min_docs].index

    df_filtered = df_filtered[df_filtered["topic_id"].isin(valid_topics)].copy()

    unique_topics = sorted(df_filtered["topic_id"].unique())
    topic_mapping = {old_id: new_idx for new_idx, old_id in enumerate(unique_topics)}

    df_filtered["original_topic_id"] = df_filtered["topic_id"]
    df_filtered["topic_id"] = df_filtered["topic_id"].map(topic_mapping)

    return df_filtered


def compute_topic_words_ctfidf(assignment_df: pd.DataFrame, texts: list,
                                max_df: float = 0.8, top_n: int = 10) -> dict:
    """Extract topic words using c-TF-IDF with parametric max_df."""
    topic_docs = []
    tids = []

    for tid, grp in assignment_df.groupby("topic_id"):
        if tid == -1:
            continue
        combined_text = " ".join([texts[i] for i in grp["doc_idx"].tolist() if i < len(texts)])
        topic_docs.append(combined_text)
        tids.append(tid)

    if not topic_docs:
        return {}

    vec = TfidfVectorizer(
        stop_words="english",
        max_features=10000,
        ngram_range=(1, 2),
        max_df=max_df
    )

    tfidf_matrix = vec.fit_transform(topic_docs)
    feature_names = vec.get_feature_names_out()

    topic_words = {}
    for i, tid in enumerate(tids):
        row = tfidf_matrix.getrow(i).toarray().flatten()
        top_ids = row.argsort()[-top_n:][::-1]
        topic_words[tid] = [feature_names[idx] for idx in top_ids if row[idx] > 0]
    return topic_words

def rbo(list1, list2, p=0.9):
    if not list1 and not list2:
        return 1.0
    if not list1 or not list2:
        return 0.0

    # assign short (S) and long (L)
    if len(list1) <= len(list2):
        S, L = list1, list2
    else:
        S, L = list2, list1

    s, l = len(S), len(L)

    S_seen = set()
    L_seen = set()

    X = 0  # overlap
    rbo = 0.0
    disjoint = 0.0
    ext_term = 0.0

    for d in range(l):
        if d < s:
            s_item = S[d]
            S_seen.add(s_item)
        else:
            s_item = None

        l_item = L[d]
        L_seen.add(l_item)

        overlap_incr = 0

        if d < s:
            if s_item == l_item:
                overlap_incr = 1
            else:
                if s_item in L_seen:
                    overlap_incr += 1
                if l_item in S_seen:
                    overlap_incr += 1
        else:
            if l_item in S_seen:
                overlap_incr = 1

        X += overlap_incr

        if d < s:
            A_d = 2.0 * X / (len(S_seen) + len(L_seen))
        else:
            A_d = X / (d + 1)

        rbo += (1 - p) * (p ** d) * A_d

        if d < s:
            ext_term = A_d * (p ** (d + 1))
        else:
            X_s = X - overlap_incr if d == s else X_s
            disjoint += (1 - p) * (p ** d) * (
                X_s * (d + 1 - s) / ((d + 1) * s)
            )
            ext_term = (
                ((X - X_s) / (d + 1) + X_s / s)
                * (p ** (d + 1))
            )

        # optional optimization (safe)
        if p ** d < 1e-12:
            break

    return min(max(rbo + disjoint + ext_term, 0.0), 1.0)

def compute_coherence_irbo(assignment_df: pd.DataFrame, texts: list,
                           max_df: float = 0.8, top_n: int = 10) -> dict:
    """Compute C_v coherence and IRBO diversity with parametric max_df."""
    topic_words = compute_topic_words_ctfidf(assignment_df, texts, max_df, top_n)
    word_lists  = [v for v in topic_words.values() if v]

    tokenized = [t.lower().split() for t in texts]

    # Flatten bigrams to unigrams for gensim
    gensim_word_lists = []
    for words in word_lists:
        topic_unigrams = []
        for w in words:
            topic_unigrams.extend(w.split())
        unique_unigrams = list(dict.fromkeys(topic_unigrams))[:top_n]
        gensim_word_lists.append(unique_unigrams)

    try:
        dct = Dictionary(tokenized)
        cm  = CoherenceModel(
            topics=gensim_word_lists,
            texts=tokenized,
            dictionary=dct,
            coherence="c_v",
            processes=5
        )
        cv = cm.get_coherence()
    except Exception as e:
        print(f"    Coherence error: {e}")
        cv = 0.0
    def irbo(lists1, lists2, p = 0.9):
        return 1 - rbo(lists1, lists2)



    pairs = list(combinations(word_lists, 2))
    irbo  = float(np.mean([irbo(a, b) for a, b in pairs])) if pairs else 0.0
    tq    = 2 * cv * irbo / (cv + irbo + 1e-8)

    return {"coherence": cv, "irbo": irbo, "topic_quality": tq}

## Topic Counts Recap

Display all topics with their document counts for all subjects before the tuning loop.

In [7]:
for subject, df_assign in all_base_assignments.items():
    print(f"=== {subject.upper()} ===")
    topic_counts = df_assign["topic_id"].value_counts().sort_values(ascending=False)
    display(pd.DataFrame({'topic': topic_counts.index, 'doc_count': topic_counts.values}))
    print("\n")

=== CS ===


,topic,doc_count
0,-1,68006
1,432,1967
2,573,1724
3,546,1704
4,578,1533
...,...,...
612,345,2
613,63,2
614,453,1
615,104,1




=== MATH ===


,topic,doc_count
0,-1,76778
1,519,1339
2,101,945
3,43,848
4,439,777
...,...,...
606,309,2
607,449,1
608,113,1
609,105,1




=== PHYSICS ===


,topic,doc_count
0,-1,92099
1,209,1358
2,50,1110
3,406,856
4,68,777
...,...,...
493,398,1
494,237,1
495,174,1
496,433,1


## Tuning Loop

Grid search over `max_df × min_docs` for each subject.
Reuses the base assignment DataFrames from modeling checkpoints.

In [8]:
all_tuning_results = []

for subject in LIST_SUBJECT:
    if subject not in all_base_assignments:
        print(f"\nSkipping {subject} (no base assignment)")
        continue

    print(f"\n{'='*60}")
    print(f"TUNING: {subject.upper()}")
    print(f"{'='*60}")

    base_df = all_base_assignments[subject]
    texts = all_data[subject]["text"].fillna("").tolist()
    best_model = BEST_MODEL_MAP[subject]

    combos = list(product(MAX_DF_VALUES, MIN_DOCS_VALUES))
    best_tq, best_config = -1, None
    previous_n_topics = 0
    previous_metrics = {}
    prev_max_df = 0.5

    for max_df, min_docs in tqdm(combos, desc=f"Tuning {subject}"):
        ckpt_name = f"tuning_{best_model}_maxdf{max_df}_mindocs{min_docs}"

        # Check if already computed
        ckpt = load_checkpoint(ckpt_name, subject)
        if ckpt:
            metrics = ckpt["metrics"]
            n_topics = ckpt["n_topics"]
        else:
            # Apply filtering with current min_docs
            df_filtered = filter_and_reindex_topics(base_df, min_docs=min_docs)
            n_topics = df_filtered["topic_id"].nunique()

            if n_topics == 0:
                print(f"  max_df={max_df}, min_docs={min_docs}: 0 topics, skipping")
                metrics = {"coherence": 0.0, "irbo": 0.0, "topic_quality": 0.0}
            else:
                # Compute coherence with current max_df
                if previous_n_topics == n_topics and prev_max_df == max_df:
                    metrics = previous_metrics
                else:
                    metrics = compute_coherence_irbo(df_filtered, texts, max_df=max_df)
                previous_n_topics = n_topics
                previous_metrics = metrics
                prev_max_df = max_df

            save_checkpoint(
                {"metrics": metrics, "n_topics": n_topics,
                 "max_df": max_df, "min_docs": min_docs},
                ckpt_name, subject
            )

        row = {
            "subject": subject,
            "best_model": best_model,
            "max_df": max_df,
            "min_docs": min_docs,
            "n_topics": n_topics,
            "coherence": metrics["coherence"],
            "irbo": metrics["irbo"],
            "topic_quality": metrics["topic_quality"],
        }
        all_tuning_results.append(row)

        if metrics["topic_quality"] > best_tq:
            best_tq = metrics["topic_quality"]
            best_config = row

        tqdm.write(
            f"  max_df={max_df:.1f}  min_docs={min_docs:3d}  "
            f"topics={n_topics:3d}  C_v={metrics['coherence']:.4f}  "
            f"IRBO={metrics['irbo']:.4f}  TQ={metrics['topic_quality']:.4f}"
        )

    print(f"\n  BEST for {subject}: max_df={best_config['max_df']}, "
          f"min_docs={best_config['min_docs']}, TQ={best_tq:.4f}")

tuning_df = pd.DataFrame(all_tuning_results)
print(f"\nTotal results: {len(tuning_df)}")


TUNING: CS


Tuning cs:   2%|▏         | 1/42 [01:46<1:13:02, 106.90s/it]

  Checkpoint saved: ../../../../models/topicGpt/cs/tuning_sentence_transformers_all_MiniLM_L6_v2_maxdf0.5_mindocs10.pkl
  max_df=0.5  min_docs= 10  topics=576  C_v=0.6227  IRBO=0.9920  TQ=0.7651


Tuning cs:   5%|▍         | 2/42 [03:26<1:08:28, 102.71s/it]

  Checkpoint saved: ../../../../models/topicGpt/cs/tuning_sentence_transformers_all_MiniLM_L6_v2_maxdf0.5_mindocs25.pkl
  max_df=0.5  min_docs= 25  topics=508  C_v=0.6348  IRBO=0.9918  TQ=0.7741


...

  Checkpoint saved: ../../../../models/topicGpt/physics/tuning_all_distilroberta_v1_maxdf1.0_mindocs150.pkl
  max_df=1.0  min_docs=150  topics=101  C_v=0.6871  IRBO=0.9658  TQ=0.8030


Tuning physics:  98%|█████████▊| 41/42 [34:42<00:45, 45.59s/it]

  Checkpoint saved: ../../../../models/topicGpt/physics/tuning_all_distilroberta_v1_maxdf1.0_mindocs200.pkl
  max_df=1.0  min_docs=200  topics= 74  C_v=0.6903  IRBO=0.9658  TQ=0.8052


Tuning physics: 100%|██████████| 42/42 [35:13<00:00, 50.32s/it]

  Checkpoint saved: ../../../../models/topicGpt/physics/tuning_all_distilroberta_v1_maxdf1.0_mindocs250.pkl
  max_df=1.0  min_docs=250  topics= 50  C_v=0.6831  IRBO=0.9586  TQ=0.7977

  BEST for physics: max_df=0.7, min_docs=200, TQ=0.8458

Total results: 126


## Results Summary

In [9]:
# Save full results CSV per subject
for subject in LIST_SUBJECT:
    subj_df = tuning_df[tuning_df["subject"] == subject]
    if len(subj_df) == 0:
        continue
    out_path = RESULT_DIR / subject / "tuning_results.csv"
    subj_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

# Also save combined results
combined_path = RESULT_DIR / "tuning_results_all.csv"
tuning_df.to_csv(combined_path, index=False)
print(f"\nCombined results saved: {combined_path}")

# Show best per subject
print(f"\n{'='*80}")
print(f"BEST CONFIGURATION PER SUBJECT")
print(f"{'='*80}")
for subject in LIST_SUBJECT:
    subj_df = tuning_df[tuning_df["subject"] == subject]
    if len(subj_df) == 0:
        continue
    best = subj_df.loc[subj_df["topic_quality"].idxmax()]
    print(f"\n{subject.upper()}:")
    print(f"  Model:    {best['best_model']}")
    print(f"  max_df:   {best['max_df']}")
    print(f"  min_docs: {int(best['min_docs'])}")
    print(f"  Topics:   {int(best['n_topics'])}")
    print(f"  C_v:      {best['coherence']:.4f}")
    print(f"  IRBO:     {best['irbo']:.4f}")
    print(f"  TQ:       {best['topic_quality']:.4f}")

# Pivot table: TQ by max_df x min_docs (averaged across subjects)
print(f"\n{'='*80}")
print(f"TOPIC QUALITY HEATMAP (averaged across subjects)")
print(f"{'='*80}")
pivot = tuning_df.pivot_table(
    index="min_docs", columns="max_df",
    values="topic_quality", aggfunc="mean"
)
print(pivot.round(4))

Saved: ../../../../results/topicGpt/tunning/cs/tuning_results.csv
Saved: ../../../../results/topicGpt/tunning/math/tuning_results.csv
Saved: ../../../../results/topicGpt/tunning/physics/tuning_results.csv

Combined results saved: ../../../../results/topicGpt/tunning/tuning_results_all.csv

BEST CONFIGURATION PER SUBJECT

CS:
  Model:    sentence_transformers_all_MiniLM_L6_v2
  max_df:   0.8
  min_docs: 200
  Topics:   138
  C_v:      0.7095
  IRBO:     0.9893
  TQ:       0.8264

MATH:
  Model:    sentence_transformers_all_MiniLM_L6_v2
  max_df:   0.9
  min_docs: 250
  Topics:   90
  C_v:      0.7280
  IRBO:     0.9809
  TQ:       0.8358

PHYSICS:
  Model:    all_distilroberta_v1
  max_df:   0.7
  min_docs: 200
  Topics:   74
  C_v:      0.7398
  IRBO:     0.9873
  TQ:       0.8458

TOPIC QUALITY HEATMAP (averaged across subjects)
max_df       0.5     0.6     0.7     0.8     0.9     1.0
min_docs                                                
10        0.7649  0.7692  0.7722  0.7718  0.

## Save Best Tuning Model

For each subject, save the best `(max_df, min_docs)` assignment + config
as a checkpoint and export the final assignment CSV.

In [10]:
for subject in LIST_SUBJECT:
    if subject not in all_base_assignments:
        continue

    subj_df = tuning_df[tuning_df["subject"] == subject]
    if len(subj_df) == 0:
        continue

    best = subj_df.loc[subj_df["topic_quality"].idxmax()]
    best_max_df = best["max_df"]
    best_min_docs = int(best["min_docs"])
    best_model = best["best_model"]

    print(f"\n{'='*60}")
    print(f"SAVING BEST FOR: {subject.upper()}")
    print(f"  model={best_model}, max_df={best_max_df}, min_docs={best_min_docs}")
    print(f"{'='*60}")

    # Re-apply filtering with best params
    base_df = all_base_assignments[subject]
    df_best = filter_and_reindex_topics(base_df, min_docs=best_min_docs)

    # Recompute metrics for verification
    texts = all_data[subject]["text"].fillna("").tolist()
    metrics = compute_coherence_irbo(df_best, texts, max_df=best_max_df)

    print(f"  Verified: C_v={metrics['coherence']:.4f}  "
          f"IRBO={metrics['irbo']:.4f}  TQ={metrics['topic_quality']:.4f}")
    print(f"  Topics: {df_best['topic_id'].nunique()}  Docs: {len(df_best)}")

    # Save best model checkpoint
    save_checkpoint(
        {
            "assignment_df": df_best,
            "metrics": metrics,
            "config": {
                "best_model": best_model,
                "max_df": best_max_df,
                "min_docs": best_min_docs,
            }
        },
        "tuning_best", subject
    )

    # Export assignment CSV
    df_export = df_best.copy()
    df_export["subject"]    = subject
    df_export["best_model"] = best_model
    df_export["max_df"]     = best_max_df
    df_export["min_docs"]   = best_min_docs
    df_export["coherence"]  = metrics["coherence"]
    df_export["irbo"]       = metrics["irbo"]

    out_csv = RESULT_DIR / subject / "topicgpt_assignments.csv"
    df_export.to_csv(out_csv, index=False)
    print(f"  Saved CSV: {out_csv}")
    print(f"    Columns: {list(df_export.columns)}")
    print(f"    Rows: {len(df_export)}")


SAVING BEST FOR: CS
  model=sentence_transformers_all_MiniLM_L6_v2, max_df=0.8, min_docs=200
  Verified: C_v=0.7095  IRBO=0.9893  TQ=0.8264
  Topics: 138  Docs: 65516
  Checkpoint saved: ../../../../models/topicGpt/cs/tuning_best.pkl
  Saved CSV: ../../../../results/topicGpt/tunning/cs/topicgpt_assignments.csv
    Columns: ['doc_idx', 'topic_id', 'topic_label', 'confidence', 'original_topic_id', 'subject', 'best_model', 'max_df', 'min_docs', 'coherence', 'irbo']
    Rows: 65516

SAVING BEST FOR: MATH
  model=sentence_transformers_all_MiniLM_L6_v2, max_df=0.9, min_docs=250
  Verified: C_v=0.7280  IRBO=0.9809  TQ=0.8358
  Topics: 90  Docs: 39176
  Checkpoint saved: ../../../../models/topicGpt/math/tuning_best.pkl
  Saved CSV: ../../../../results/topicGpt/tunning/math/topicgpt_assignments.csv
    Columns: ['doc_idx', 'topic_id', 'topic_label', 'confidence', 'original_topic_id', 'subject', 'best_model', 'max_df', 'min_docs', 'coherence', 'irbo']
    Rows: 39176

SAVING BEST FOR: PHYSICS
 

## Final Summary

In [11]:
print(f"\n{'='*80}")
print(f"{'SUBJECT':<12} | {'MODEL':<45} | {'max_df':>6} | {'min_docs':>8} | {'TOPICS':>6} | {'TQ':>6}")
print(f"{'-'*80}")

for subject in LIST_SUBJECT:
    subj_df = tuning_df[tuning_df["subject"] == subject]
    if len(subj_df) == 0:
        continue
    best = subj_df.loc[subj_df["topic_quality"].idxmax()]
    print(f"{subject:<12} | {best['best_model']:<45} | {best['max_df']:>6.1f} | {int(best['min_docs']):>8} | {int(best['n_topics']):>6} | {best['topic_quality']:>6.4f}")

print(f"\nDone! All results saved to {RESULT_DIR.resolve()}")
print(f"Best model checkpoints saved to {CHECKPOINT_DIR.resolve()}/{{subject}}/tuning_best.pkl")


SUBJECT      | MODEL                                         | max_df | min_docs | TOPICS |     TQ
--------------------------------------------------------------------------------
cs           | sentence_transformers_all_MiniLM_L6_v2        |    0.8 |      200 |    138 | 0.8264
math         | sentence_transformers_all_MiniLM_L6_v2        |    0.9 |      250 |     90 | 0.8358
physics      | all_distilroberta_v1                          |    0.7 |      200 |     74 | 0.8458

Done! All results saved to /home/nedo/Kuliah/TA/Program/results/topicGpt/tunning
Best model checkpoints saved to /home/nedo/Kuliah/TA/Program/models/topicGpt/{subject}/tuning_best.pkl
